### **Code : Report 작성을 위한 텍스트 → 숫자 변환**
#### Writer : Donghyeon Kim
#### Update : 2026.01.16.

---

#### **Memo**
#### **보고서 작성을 위한 텍스트 응답 숫자화 및 순위표 생성**

본 Python Notebook은 본설문 원자료의 주요 텍스트 응답을 숫자 코드로 변환하고, P2-Q5/P2-Q6 복수 선택형 텍스트 응답을 순위형 빈도표로 정리하는 파일이다.  
원본 컬럼은 보존하고, 변환 결과는 별도 컬럼과 Excel/CSV 파일로 저장한다.

---

#### **1. 기본 설정**

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd


def resolve_project_paths(start: Path | None = None) -> tuple[Path, Path, Path]:
    """현재 실행 위치가 달라도 분석/보고서 폴더를 안정적으로 찾아내는 함수"""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()

    for base in (start, *start.parents):
        if (base / "1. Rawdata").exists() and (base / "2. Python Code").exists():
            analysis_dir = base
            report_dir = base.parent / "3. 보고서"
            output_dir = report_dir / "Python Results"
            output_dir.mkdir(parents=True, exist_ok=True)
            return analysis_dir, report_dir, output_dir

        if (base / "2. 분석").exists() and (base / "3. 보고서").exists():
            analysis_dir = base / "2. 분석"
            report_dir = base / "3. 보고서"
            output_dir = report_dir / "Python Results"
            output_dir.mkdir(parents=True, exist_ok=True)
            return analysis_dir, report_dir, output_dir

    raise FileNotFoundError("분석/보고서 폴더를 찾지 못했습니다. 노트북 실행 위치를 확인하세요.")


ANALYSIS_DIR, REPORT_DIR, OUTPUT_DIR = resolve_project_paths()

RAW_FILE = REPORT_DIR / "0_본설문_Rawdata_통합_분석용.xlsx"
CLUSTER_FILE = REPORT_DIR / "2_TimeCost_본설문_Rawdata_Cluster3.xlsx"

CONVERTED_OUTPUT = OUTPUT_DIR / "converted_result.csv"
RANK_OUTPUT = OUTPUT_DIR / "survey_rank_frequency.xlsx"
CLUSTER_RANK_OUTPUT = OUTPUT_DIR / "Cluster_Item_Rank_Frequency.xlsx"

print(f"분석 폴더: {ANALYSIS_DIR}")
print(f"보고서 폴더: {REPORT_DIR}")
print(f"결과 저장 폴더: {OUTPUT_DIR}")

분석 폴더: /Users/hyeondk/Dropbox/6. C&S Lab/8. 2025년/2. [한생연] 바이오소재 경제가치/2. 분석
보고서 폴더: /Users/hyeondk/Dropbox/6. C&S Lab/8. 2025년/2. [한생연] 바이오소재 경제가치/3. 보고서
결과 저장 폴더: /Users/hyeondk/Dropbox/6. C&S Lab/8. 2025년/2. [한생연] 바이오소재 경제가치/3. 보고서/Python Results


---

#### **2. 텍스트 응답 숫자 코드 변환**

In [2]:
benefit_map = {
    "이용해 보지 않았다": 1,
    "전혀 혜택이 없다": 2,
    "혜택이 매우 높다": 6,
    "혜택이 높다": 5,
    "혜택이 낮다": 3,
    "혜택 낮다": 3,
    "보통이다": 4
}

time_cost_map = {
    "1~6시간": 1,
    "7~12시간": 2,
    "13~24시간": 3,
    "12~24시간": 3,
    "1~3일": 4,
    "4~7일": 5,
    "3~7일": 5,
    "8~15일": 6,
    "16일~1개월": 7,
    "1개월": 7,
    "2~6개월": 8,
    "6개월": 8,
    "6~12개월": 9,
    "1년 이상": 10
}

efficiency_map = {
    "없음": 1,
    "5%": 2,
    "10%": 3,
    "20%": 4,
    "30%": 5,
    "40%": 6,
    "50%": 7,
    "60%": 8,
    "70%": 9,
    "80%": 10,
    "90%": 11,
    "100%": 12
}

yes_no_map = {"예": 1, "아니오": 2}

CODEBOOKS = {
    "P1-Q1": {"남자": 1, "여자": 2},
    "P1-Q2": {"21~29세": 1, "30~39세": 2, "40~49세": 3, "50~59세": 4, "60세 이상": 5},
    "P1-Q3": {
        "대학": 1,
        "기업": 2,
        "연구기관(출연연, 국공립 등)": 3,
        "연구기관(출연(연), 국공립(연) 등)": 3,
        "기타": 4
    },
    "P1-Q5": {"박사": 1, "석사": 2, "학사": 3, "전문학사": 4, "기타": 5},
    "P1-Q6": {"과제책임자": 1, "공동연구원": 2, "기타": 3},
    "P2-Q1": {
        "컴퓨터 작업(정보 및 데이터 수집, 처리 등)": 1,
        "컴퓨터 작업": 1,
        "실험실 작업(실험 수행 및 모니터링, 임상실험 등)": 2,
        "실험실 작업": 2,
        "서류 작업(문헌 정리, 실험결과 정리 등)": 3,
        "서류작업(문헌정리, 실험결과 정리 등)": 3,
        "서류 작업": 3,
        "기타": 4
    },
    "P3-Q2": time_cost_map,
    "P3-Q4": time_cost_map,
    "P3-Q5-1": benefit_map,
    "P3-Q5-2": benefit_map,
    "P3-Q5-3": benefit_map,
    "P3-Q5-4": benefit_map,
    "P3-Q5-5": benefit_map,
    "P3-Q5-6": benefit_map,
    "P3-Q10": efficiency_map,
    "P3-Q11": efficiency_map,
    "P4-Q1": yes_no_map,
    "P4-Q5": yes_no_map,
    "P5-Q1": yes_no_map,
    "P5-Q2": yes_no_map,
    "P5-Q3": yes_no_map
}


def normalize_text(value) -> str:
    """선택지 번호, 줄바꿈, 중복 공백을 줄여 비교용 문자열로 생성"""
    text = str(value).strip()
    text = re.sub(r"^[①②③④⑤⑥⑦⑧⑨⑩]\s*", "", text)
    text = re.sub(r"^\d+\s*(?:[.)]|번)\s*", "", text)
    text = re.sub(r"^\d+\s+(?=\S)", "", text)
    return re.sub(r"\s+", " ", text)


def map_text_to_code(value, codebook: dict[str, int]):
    """정확 일치 후 긴 선택지부터 포함 여부를 확인해 코드로 변환"""
    if pd.isna(value):
        return pd.NA

    if isinstance(value, (int, np.integer)) and int(value) in set(codebook.values()):
        return int(value)

    text = normalize_text(value)
    if text in codebook:
        return codebook[text]

    for label in sorted(codebook, key=len, reverse=True):
        if label in text:
            return codebook[label]

    return pd.NA


def add_code_columns(df: pd.DataFrame, codebooks: dict[str, dict[str, int]]) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """원본 컬럼은 유지하고 `{컬럼명}_Code` 컬럼을 추가"""
    result = df.copy()
    unmatched = {}

    for col, codebook in codebooks.items():
        if col not in result.columns:
            print(f"[건너뜀] {col}: 원자료에 컬럼 없음")
            continue

        converted = result[col].apply(lambda value: map_text_to_code(value, codebook))
        result[f"{col}_Code"] = converted.astype("Int64")

        miss_mask = converted.isna() & result[col].notna()
        if miss_mask.any():
            unmatched[col] = sorted(result.loc[miss_mask, col].astype(str).unique())

    return result, unmatched

In [3]:
if not RAW_FILE.exists():
    raise FileNotFoundError(f"원자료 파일을 찾지 못했습니다: {RAW_FILE}")

df = pd.read_excel(RAW_FILE)
df_converted, unmatched_values = add_code_columns(df, CODEBOOKS)
df_converted.to_csv(CONVERTED_OUTPUT, index=False, encoding="utf-8-sig")

print(f"원자료 행/열: {df.shape}")
print(f"변환 후 행/열: {df_converted.shape}")
print(f"저장 완료: {CONVERTED_OUTPUT}")

if unmatched_values:
    print("\n[확인 필요] 매핑되지 않은 값")
    for col, values in unmatched_values.items():
        preview = ", ".join(values[:5])
        suffix = " ..." if len(values) > 5 else ""
        print(f"- {col}: {preview}{suffix}")
else:
    print("모든 지정 컬럼에 대해 코드 변환 완료")

원자료 행/열: (154, 46)
변환 후 행/열: (154, 67)
저장 완료: /Users/hyeondk/Dropbox/6. C&S Lab/8. 2025년/2. [한생연] 바이오소재 경제가치/3. 보고서/Python Results/converted_result.csv
모든 지정 컬럼에 대해 코드 변환 완료


---

#### **3. 정보 항목 순위 응답 변환 및 빈도표 생성**

In [4]:
ITEMS_MAP = {
    "소재관련 정보(기본/특성 정보, 분양 정보)": "소재정보",
    "소재관련 연관정보 및 추천 정보": "연관정보",
    "논문정보(검색 소재와 유사성 높은 국내외 논문의 저널, 초록, 키워드, 보유 소재 등)": "논문정보",
    "특허정보(검색 소재와 유사성 높은 국내외 특허의 출원, 초록, 키워드, 보유 소재 등)": "특허정보",
    "신약정보(검색 소재와 유사성 높은 질병, 타겟, 약물 등)": "신약정보",
    "통합플랫폼 콘텐츠(자료실, 동향, 통계 등)": "콘텐츠"
}
ITEM_NAMES = list(ITEMS_MAP.values())
RANK_COLUMNS = list(range(1, len(ITEM_NAMES) + 1))


def extract_item_ranks(text, items_map: dict[str, str] = ITEMS_MAP) -> dict[str, float]:
    """응답 텍스트에서 항목이 등장한 순서를 1~6위로 변환"""
    ranks = {short_name: np.nan for short_name in items_map.values()}
    if pd.isna(text):
        return ranks

    text = str(text)
    found_items = []
    for full_name, short_name in items_map.items():
        idx = text.find(full_name)
        if idx >= 0:
            found_items.append((idx, short_name))

    for rank, (_, short_name) in enumerate(sorted(found_items), start=1):
        ranks[short_name] = rank

    return ranks


def make_rank_dataframe(df: pd.DataFrame, col: str, prefix: str | None = None) -> pd.DataFrame:
    if col not in df.columns:
        raise KeyError(f"원자료에 {col} 컬럼이 없습니다.")

    rank_df = pd.DataFrame(df[col].apply(extract_item_ranks).tolist(), index=df.index)
    rank_df = rank_df.reindex(columns=ITEM_NAMES)
    if prefix:
        rank_df = rank_df.add_prefix(f"{prefix}_")
    return rank_df


def make_frequency_table(rank_df: pd.DataFrame) -> pd.DataFrame:
    table = pd.DataFrame(0, index=rank_df.columns, columns=RANK_COLUMNS, dtype=int)
    for item in rank_df.columns:
        counts = rank_df[item].value_counts(dropna=True)
        for rank in RANK_COLUMNS:
            table.loc[item, rank] = int(counts.get(rank, 0))
    table["Total"] = table.sum(axis=1)
    return table

In [5]:
ranks_q5 = make_rank_dataframe(df, "P2-Q5")
ranks_q6 = make_rank_dataframe(df, "P2-Q6")
freq_q5 = make_frequency_table(ranks_q5)
freq_q6 = make_frequency_table(ranks_q6)

with pd.ExcelWriter(RANK_OUTPUT) as writer:
    freq_q5.to_excel(writer, sheet_name="Q5_빈도표")
    freq_q6.to_excel(writer, sheet_name="Q6_빈도표")
    ranks_q5.to_excel(writer, sheet_name="Q5_개별순위데이터", index=False)
    ranks_q6.to_excel(writer, sheet_name="Q6_개별순위데이터", index=False)

print("--- Q5 (자주 이용) 빈도표 ---")
display(freq_q5)
print("--- Q6 (중요도) 빈도표 ---")
display(freq_q6)
print(f"저장 완료: {RANK_OUTPUT}")

--- Q5 (자주 이용) 빈도표 ---


,1,2,3,4,5,6,Total
소재정보,79,18,16,19,9,13,154
연관정보,15,50,31,23,25,10,154
논문정보,31,37,35,20,25,6,154
특허정보,11,18,22,40,44,19,154
신약정보,7,21,29,22,33,42,154
콘텐츠,11,10,21,30,18,64,154


--- Q6 (중요도) 빈도표 ---


,1,2,3,4,5,6,Total
소재정보,56,25,23,27,10,13,154
연관정보,19,52,33,19,20,11,154
논문정보,37,30,41,28,15,3,154
특허정보,9,21,22,36,32,34,154
신약정보,19,16,22,23,39,35,154
콘텐츠,14,10,13,21,38,58,154


저장 완료: /Users/hyeondk/Dropbox/6. C&S Lab/8. 2025년/2. [한생연] 바이오소재 경제가치/3. 보고서/Python Results/survey_rank_frequency.xlsx


---

#### **4. 군집별 정보 항목 순위 빈도표 생성**

In [6]:
def format_cluster_label(value) -> str:
    """군집 번호가 1/2/3 또는 문자열이어도 보고서 표기 형태로 맞춤"""
    if pd.isna(value):
        return "Cluster 없음"

    text = str(value).strip()
    if text.lower().startswith("cluster"):
        return text

    numeric = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    if pd.notna(numeric):
        return f"Cluster {int(numeric)}"
    return text


def cluster_sort_key(label: str) -> tuple[int, str]:
    match = re.search(r"\d+", str(label))
    return (int(match.group()) if match else 9999, str(label))


def add_rank_columns(df: pd.DataFrame) -> pd.DataFrame:
    q5_ranks = make_rank_dataframe(df, "P2-Q5", prefix="Q5")
    q6_ranks = make_rank_dataframe(df, "P2-Q6", prefix="Q6")
    return pd.concat([df.reset_index(drop=True), q5_ranks.reset_index(drop=True), q6_ranks.reset_index(drop=True)], axis=1)


def make_prefixed_frequency_table(subset_df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    rank_cols = [f"{prefix}_{item}" for item in ITEM_NAMES]
    rank_df = subset_df[rank_cols].copy()
    rank_df.columns = ITEM_NAMES
    return make_frequency_table(rank_df)


def export_cluster_rank_report(df_cluster: pd.DataFrame, output_path: Path) -> pd.DataFrame:
    if "Cluster" not in df_cluster.columns:
        raise KeyError("Cluster 컬럼이 없습니다. 먼저 군집분석 노트북에서 군집 결과 파일을 생성하세요.")

    report_df = add_rank_columns(df_cluster)
    report_df["Cluster_Label"] = report_df["Cluster"].apply(format_cluster_label)
    cluster_labels = sorted(report_df["Cluster_Label"].dropna().unique(), key=cluster_sort_key)

    with pd.ExcelWriter(output_path) as writer:
        row_idx = 0
        for label in cluster_labels:
            subset = report_df[report_df["Cluster_Label"] == label]
            pd.DataFrame([f"■ {label} (인원: {len(subset)}명)"]).to_excel(
                writer, sheet_name="Report", startrow=row_idx, index=False, header=False
            )
            row_idx += 2

            for q_code, q_desc in (("Q5", "자주 이용 정보"), ("Q6", "중요 정보")):
                pd.DataFrame([f"[{q_desc} - {q_code}]"]).to_excel(
                    writer, sheet_name="Report", startrow=row_idx, index=False, header=False
                )
                row_idx += 1

                table = make_prefixed_frequency_table(subset, q_code)
                table.to_excel(writer, sheet_name="Report", startrow=row_idx)
                row_idx += len(table) + 3

            row_idx += 2

        report_df.to_excel(writer, sheet_name="RawData", index=False)

    return report_df

In [7]:
if CLUSTER_FILE.exists():
    cluster_df = pd.read_excel(CLUSTER_FILE)
    cluster_report_df = export_cluster_rank_report(cluster_df, CLUSTER_RANK_OUTPUT)
    print(f"군집 자료 행/열: {cluster_df.shape}")
    print(f"저장 완료: {CLUSTER_RANK_OUTPUT}")
    display(cluster_report_df["Cluster_Label"].value_counts().sort_index())
else:
    print(f"군집 결과 파일이 없어 군집별 빈도표 생성을 건너뜁니다: {CLUSTER_FILE}")

군집 자료 행/열: (117, 58)
저장 완료: /Users/hyeondk/Dropbox/6. C&S Lab/8. 2025년/2. [한생연] 바이오소재 경제가치/3. 보고서/Python Results/Cluster_Item_Rank_Frequency.xlsx


Cluster_Label
Cluster 1     64
Cluster 2     37
Cluster 3     13
Cluster 없음     3
Name: count, dtype: int64